# Week 52, Day 6: Part 2 exam

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w052/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: B1: excitation, inhibition and division

A single compartment has a leak (gL = 10 nS, EL = -70 mV), excitation (EE = 0 mV) and inhibition (EI = -70 mV, at rest). Write v_ss(gE, gI, EI=-70.0), the steady-state voltage. For each gE in the list, compute the ratio of the depolarisation from rest with gI = 10 nS to that with gI = 0.

In [ ]:
import numpy as np
gL, EL, EE = 10.0, -70.0, 0.0

def v_ss(gE, gI, EI=-70.0):
    return ...

gEs = np.array([0.5, 2.0, 5.0, 10.0])
ratio = ...
for gE, rr in zip(gEs, ratio):
    print(f"gE = {gE:4.1f} nS: depolarisation {v_ss(gE, 0) - EL:5.2f} mV without inhibition, {v_ss(gE, 10) - EL:5.2f} mV with; ratio {rr:.3f}")
print(f"inhibition at -80 mV alone: {v_ss(0, 10, EI=-80):.1f} mV")

In [ ]:
import numpy as np
def test_vss():
    "v_ss is the conductance-weighted average of reversal potentials"
    assert abs(v_ss(5.0, 10.0) - (10 * -70 + 5 * 0 + 10 * -70) / 25) < 1e-12
def test_divide():
    "with shunting inhibition the depolarisation ratio stays within 0.05 of gL / (gL + gI) for small excitation"
    assert abs(ratio[0] - 10 / 20) < 0.05
def test_hyper():
    "hyperpolarising inhibition (E_I = -80 mV) lowers the voltage below rest without excitation"
    assert v_ss(0.0, 10.0, EI=-80.0) < -70

test_vss()
test_divide()
test_hyper()
print("All checks passed")

### Exercise: B2: is the choice in the population?

Eighty neurons' trial counts depend weakly on an upcoming choice. Write cv_acc(X, y), the 5-fold cross-validated accuracy of a ridge logistic decoder (folds by trial order), then compare the real accuracy with 100 accuracies after shuffling the labels.

In [ ]:
g = np.random.default_rng(52)
n_tr, n_n = 300, 80
y = g.integers(0, 2, n_tr).astype(float)
effect = g.normal(0, 0.25, n_n)
X = g.poisson(np.exp(1.5 + np.outer(2 * y - 1, effect) * 0.2)).astype(float)
X = (X - X.mean(0)) / (X.std(0) + 1e-9)

def fit_logreg(X, y, lam=1.0):
    w = np.zeros(X.shape[1] + 1); A = np.c_[X, np.ones(len(y))]
    for _ in range(20):
        p = 1 / (1 + np.exp(-A @ w))
        H = -(A * (p * (1 - p))[:, None]).T @ A - lam * np.eye(len(w)); w -= np.linalg.solve(H, A.T @ (y - p) - lam * w)
    return w

def cv_acc(X, y, k=5):
    folds = np.array_split(np.arange(len(y)), k); correct = 0
    for f in folds:
        tr = np.setdiff1d(np.arange(len(y)), f)
        w = fit_logreg(X[tr], y[tr])
        correct += ...
    return correct / len(y)

cv_accuracy = cv_acc(X, y)
null = np.array([cv_acc(X, g.permutation(y)) for _ in range(100)])
p_value = ...
print(f"cross-validated accuracy {cv_accuracy:.3f}; shuffled labels {null.mean():.3f} (95th percentile {np.percentile(null, 95):.3f}); p = {p_value:.4f}")

In [ ]:
import numpy as np
def test_cv():
    "cv_accuracy uses 5 folds and lies between 0.6 and 0.9"
    assert 0.6 < cv_accuracy < 0.9
def test_null():
    "null holds 100 shuffled-label accuracies centred near 0.5"
    assert len(null) == 100 and abs(np.mean(null) - 0.5) < 0.03
def test_p():
    "p_value is the fraction of null accuracies at least as large as the real one, with the +1 correction"
    assert abs(p_value - (1 + np.sum(null >= cv_accuracy)) / 101) < 1e-12 and p_value < 0.02

test_cv()
test_null()
test_p()
print("All checks passed")

### Exercise: B3: when does STDP strengthen a synapse?

Write the STDP window stdp(dt) with A_plus = 0.01, A_minus = 0.0105 and tau = 20 ms for both sides, where dt = t_post - t_pre. If the timing differences are Gaussian with mean mu and SD 10 ms, write expected_dw(mu) by numerical integration, and find with brentq the mean timing mu0 at which the expected change is zero.

In [ ]:
from scipy.optimize import brentq
A_plus, A_minus, tau, sd = 0.01, 0.0105, 20.0, 10.0
d = np.linspace(-150, 150, 30001)

def stdp(dt):
    return ...

def expected_dw(mu):
    pdf = np.exp(-0.5 * ((d - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
    return ...

mu0 = brentq(expected_dw, -20, 20)
for mu in [-20, -5, 0, 5, 20]:
    print(f"mean timing {mu:4d} ms: expected weight change {expected_dw(mu):+.5f}")
print(f"break-even mean timing: {mu0:.2f} ms")

In [ ]:
import numpy as np
def test_window():
    "stdp gives A_plus exp(-dt/tau) for positive dt (pre before post) and -A_minus exp(dt/tau) for negative"
    assert abs(stdp(np.array([10.0]))[0] - 0.01 * np.exp(-0.5)) < 1e-12 and abs(stdp(np.array([-10.0]))[0] + 0.0105 * np.exp(-0.5)) < 1e-12
def test_expected():
    "expected_dw(mu) integrates the window against a Gaussian of timing differences"
    d = np.linspace(-150, 150, 30001); pdf = np.exp(-0.5 * ((d - 5) / 10) ** 2) / (10 * np.sqrt(2 * np.pi))
    assert abs(expected_dw(5.0) - np.sum(stdp(d) * pdf) * (d[1] - d[0])) < 1e-8
def test_zero():
    "the break-even mean timing mu0 is small and positive"
    assert 0 < mu0 < 3 and abs(expected_dw(mu0)) < 1e-9

test_window()
test_expected()
test_zero()
print("All checks passed")

### Exercise: B4: a learning rate from conditioning data

An animal is rewarded on 75% of 200 trials and its anticipatory response on each trial is the Rescorla-Wagner prediction plus noise. Write rw_predictions(alpha, r), the prediction before each trial starting from zero, and find alpha_hat by minimising the squared error between predictions and responses over a fine grid of learning rates.

In [ ]:
g = np.random.default_rng(44)
r = (g.random(200) < 0.75).astype(float)

def rw_predictions(alpha, r):
    V, out = 0.0, np.zeros(len(r))
    for t in range(len(r)):
        out[t] = V
        V = ...
    return out

responses = rw_predictions(0.15, r) + g.normal(0, 0.1, 200)
grid = np.linspace(0.01, 0.6, 300)
sse = np.array([np.sum((rw_predictions(a, r) - responses) ** 2) for a in grid])
alpha_hat = ...
print(f"fitted learning rate {alpha_hat:.3f} (true 0.15); final prediction {rw_predictions(alpha_hat, r)[-1]:.2f} against a reward probability of 0.75")

In [ ]:
import numpy as np
def test_rw():
    "rw_predictions follows V <- V + alpha (r - V), returning V before each trial"
    v = rw_predictions(0.5, np.array([1.0, 1.0, 0.0]))
    assert np.allclose(v, [0, 0.5, 0.75])
def test_fit():
    "alpha_hat recovers the true learning rate within 0.05"
    assert abs(alpha_hat - 0.15) < 0.05

test_rw()
test_fit()
print("All checks passed")